# 07 - Convert HF -> GGUF untuk Ollama

**Tujuan:** Convert model ke format GGUF supaya bisa dijalankan via Ollama lokal.

**Lokasi:** VS Code (CPU task, tidak butuh GPU)

---
**Output:** `models/gguf/model.gguf` (F16)


Saved results are in `results/reference/`. Your results may differ with other inputs or settings. See `docs/results.md` for details.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src" / "config.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from inside the project directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

import subprocess
from pathlib import Path

from src.config import ABLITERATED_HF_DIR, GGUF_DIR

# Path llama.cpp converter
LLAMA_CPP_DIR = Path.home() / 'llama.cpp'
CONVERTER = LLAMA_CPP_DIR / 'convert_hf_to_gguf.py'

GGUF_DIR.mkdir(parents=True, exist_ok=True)
print(f'GGUF output: {GGUF_DIR / "model.gguf"}')
print(f'Converter: {CONVERTER}')

In [ ]:
if CONVERTER.exists():
    print(f"Converter ready: {CONVERTER}")
else:
    print("llama.cpp not found. Cloning...")
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git {LLAMA_CPP_DIR}


In [ ]:
if CONVERTER.exists():
    outpath = GGUF_DIR / 'model.gguf'
    
    # Catatan: --outtype f16 / bf16 / q8_0 aja (q4_0 tidak support di converter)
    # Quantisasi 4-bit bisa pake llama-quantize (perlu compile C++) atau pake Ollama langsung
    cmd = [
        sys.executable, str(CONVERTER),
        str(ABLITERATED_HF_DIR),
        '--outfile', str(outpath),
        '--outtype', 'f16',
    ]
    print(f'Running: {" ".join(cmd)}')
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(LLAMA_CPP_DIR))
    
    # Tampilkan progress log
    log = (result.stdout[-1000:] if result.stdout else '') + (result.stderr[-500:] if result.stderr else '')
    print(log)
    
    if result.returncode == 0 and outpath.exists():
        size_mb = outpath.stat().st_size / 1e6
        print(f'Done! GGUF file: {outpath} ({size_mb:.0f} MB)')
    else:
        raise RuntimeError(f'Converter failed: {result.stderr}')
else:
    print("Converter not available. Clone dulu: git clone https://github.com/ggerganov/llama.cpp.git ~/llama.cpp")

In [ ]:
gguf_path = GGUF_DIR / 'model.gguf'
if gguf_path.exists():
    size_mb = gguf_path.stat().st_size / 1e6
    size_gb = gguf_path.stat().st_size / 1e9
    print(f'GGUF file: {gguf_path}')
    print(f'Size: {size_gb:.1f} GB ({size_mb:.0f} MB)')
else:
    print("GGUF file not found. Run conversion cell above.")

---
**Next:** Lanjut ke `08_ollama_eval.ipynb`
